# 4-Pipeline Inpainting Comparison

Visual comparison of four inpainting methods on our report test images:
1. **Vanilla** — standard DDPM inpainting
2. **Structure-Aware** — structural edge guidance with latent gradient guidance (LGG)
3. **Sampling-Refined** — RePaint-style resampling for better coherence
4. **Full** — combines structural guidance, LGG, and RePaint resampling

In [ ]:
import sys
import json
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, '.')

from utils.cli import load_sd_pipeline
from utils.image import apply_mask_for_display
from vanilla_inpaint import ddpm_inpaint
from structure_aware_inpaint import ddpm_structural_lgg_inpaint
from sampling_refined_inpaint import ddpm_inpaint_improved
from full_inpaint import ddpm_inpaint_final

# Config
SEED = 42
STEPS = 50
GUIDANCE_SCALE = 7.5
RESAMPLE_STEPS = 5

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

pipe = load_sd_pipeline(device)

In [ ]:
# Load images, masks, and prompts from report_images/
report_dir = Path("report_images")

test_cases = []
for subdir in sorted(report_dir.iterdir()):
    if not subdir.is_dir():
        continue
    prompt_file = subdir / "prompt.json"
    mask_file = subdir / "mask.png"
    if not prompt_file.exists() or not mask_file.exists():
        print(f"  SKIPPING {subdir.name} (missing prompt.json or mask.png)")
        continue

    # Find image file (JPG/JPEG/PNG)
    img_path = None
    for name in ("image.JPG", "image.jpg", "image.jpeg", "image.png", "image.PNG"):
        candidate = subdir / name
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        print(f"  SKIPPING {subdir.name} (no image file found)")
        continue

    with open(prompt_file) as f:
        prompt = json.load(f)["prompt"]

    image = Image.open(img_path).convert("RGB").resize((512, 512))

    mask_img = Image.open(mask_file).convert("L").resize((512, 512), Image.NEAREST)
    # Invert: white (255) in PNG → 0.0 (inpaint), black (0) → 1.0 (keep)
    mask_tensor = torch.from_numpy(
        (np.array(mask_img) <= 127).astype(np.float32)
    ).unsqueeze(0).unsqueeze(0)

    test_cases.append({
        "name": subdir.name,
        "image": image,
        "mask": mask_tensor,
        "prompt": prompt,
    })

print(f"Loaded {len(test_cases)} test cases:")
for tc in test_cases:
    print(f"  {tc['name']}: \"{tc['prompt']}\"")

In [ ]:
# Run all 4 pipelines on each test case
results = []

for i, tc in enumerate(test_cases):
    print(f"[{i+1}/{len(test_cases)}] {tc['name']}: \"{tc['prompt']}\"")

    image, mask, prompt = tc["image"], tc["mask"], tc["prompt"]
    masked_display = apply_mask_for_display(image, mask)

    # 1. Vanilla
    vanilla_out = ddpm_inpaint(pipe, image, mask, prompt, STEPS, GUIDANCE_SCALE, SEED)

    # 2. Structure-Aware
    structure_out = ddpm_structural_lgg_inpaint(
        pipe, image, mask, prompt,
        steps=STEPS, guidance_scale=GUIDANCE_SCALE, seed=SEED
    )

    # 3. Sampling-Refined
    sampling_out = ddpm_inpaint_improved(
        pipe, image, mask, prompt,
        STEPS, GUIDANCE_SCALE, SEED, RESAMPLE_STEPS
    )

    # 4. Full
    full_out = ddpm_inpaint_final(
        pipe, image, mask, prompt,
        steps=STEPS, guidance_scale=GUIDANCE_SCALE,
        seed=SEED, resample_steps=RESAMPLE_STEPS
    )

    results.append({
        "image": image,
        "masked": masked_display,
        "vanilla": vanilla_out,
        "structure": structure_out,
        "sampling": sampling_out,
        "full": full_out,
        "prompt": prompt,
    })
    print(f"  Done.")

In [ ]:
# Display comparison grid: 7 columns (prompt text + 6 images)
n_images = len(results)
col_titles = ["Prompt", "Original", "Masked", "Vanilla", "Structure-Aware", "Sampling-Refined", "Full"]
n_cols = 7

fig, axes = plt.subplots(n_images, n_cols, figsize=(n_cols * 3, n_images * 3))
if n_images == 1:
    axes = axes[None, :]

for row, res in enumerate(results):
    # Column 0: prompt text
    ax = axes[row, 0]
    ax.axis("off")
    ax.text(0.5, 0.5, f'"{res["prompt"]}"',
            ha="center", va="center", fontsize=9,
            wrap=True, transform=ax.transAxes)
    if row == 0:
        ax.set_title(col_titles[0], fontsize=11, fontweight="bold")

    # Columns 1-6: images
    imgs = [res["image"], res["masked"], res["vanilla"],
            res["structure"], res["sampling"], res["full"]]
    for col, img in enumerate(imgs, start=1):
        ax = axes[row, col]
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        if row == 0:
            ax.set_title(col_titles[col], fontsize=11, fontweight="bold")

plt.tight_layout()

# Save
os.makedirs("results", exist_ok=True)
fig.savefig("results/report_4_pipelines.png", dpi=150, bbox_inches="tight")
print("Saved to results/report_4_pipelines.png")
plt.show()